# Phase 2: Stage 1 vs Stage 2 Comparison Evaluation

This notebook evaluates all Stage 1 (frozen encoder) and Stage 2 (end-to-end) checkpoints separately to generate the Phase 2 comparison table.

**What this notebook does:**
1. Loads both Stage 1 and Stage 2 checkpoints for CAE, SimCLR, and MoCo
2. Evaluates each checkpoint independently on Test.csv
3. Compares frozen vs end-to-end fine-tuning effectiveness
4. Generates comparison tables and saves results to Drive

**Run folders used:**
- `Prostate_SSL/runs/baseline_20260329_110621/`
- `Prostate_SSL/runs/cae_20260329_114825/`
- `Prostate_SSL/runs/moco_20260329_024342/`
- `Prostate_SSL/runs/simclr_20260329_122931/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from datetime import datetime

REPO_URL = 'https://github.com/satvikkaul/SSL_Prostate_Cancer_Grading.git'
BRANCH = 'method/moco-v2'  # Use the correct branch with all models
PROJECT_DIR = '/content/SSL_Prostate_Cancer_Grading'
DRIVE_ROOT = '/content/drive/MyDrive/Prostate_SSL'
DATASET_ZIP = f'{DRIVE_ROOT}/dataset.zip'
RUNS_DIR = f'{DRIVE_ROOT}/runs'

os.environ['MPLCONFIGDIR'] = '/tmp/mplconfig'

print('PROJECT_DIR  =', PROJECT_DIR)
print('DATASET_ZIP  =', DATASET_ZIP)
print('RUNS_DIR     =', RUNS_DIR)
print()
print('Available run folders:')
!ls -1 "$RUNS_DIR" | grep -E "(baseline|cae|moco|simclr)_202"

In [ ]:
%cd /content
!rm -rf "$PROJECT_DIR"
!git clone -b "$BRANCH" "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"
!git branch --show-current
!git log -1 --oneline

In [ ]:
%cd "$PROJECT_DIR"
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!python --version
!nvidia-smi

In [ ]:
# Extract dataset.zip to local SSD (faster than reading from Drive)
%cd "$PROJECT_DIR"
!test -f "$DATASET_ZIP" || (echo "Missing dataset.zip at $DATASET_ZIP" && exit 1)
!rm -rf ./dataset
!cp "$DATASET_ZIP" ./dataset.zip
!unzip -q ./dataset.zip
!rm -f ./dataset.zip
!ls ./dataset | head -5
print("\n✅ Dataset extracted to ./dataset/")

In [ ]:
# Verify dataset CSVs exist (or generate them)
%cd "$PROJECT_DIR"
import os

required_files = [
    './dataset/Train.csv',
    './dataset/Test.csv',
    './dataset/TrainSplit.csv',
    './dataset/Val.csv',
    './dataset/Pretrain_Manifest.csv',
]

missing = [path for path in required_files if not os.path.exists(path)]
if missing:
    print('Missing generated dataset files:', missing)
    !python data/setup.py
else:
    print('✅ Dataset CSVs and manifest already present.')

In [ ]:

# Import libraries and define evaluation helpers
import sys
import os
import zipfile
import json
import tempfile
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (
    classification_report,
    cohen_kappa_score,
    roc_auc_score,
    confusion_matrix,
)
from pathlib import Path
from data.generator import DataGenerator

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

# IMPORTANT: class order must match training order used in all train scripts
# train_baseline.py / train_cae.py etc. all use ['NC', 'G3', 'G5', 'G4']
# G5 is index 2, G4 is index 3  — NOT alphabetical order
CLASS_NAMES = ['NC', 'G3', 'G5', 'G4']
IMG_DIM = (128, 128, 3)


def load_model_compat(model_path):
    """
    Load a .keras model saved with Keras 3.x that contains quantization_config
    in Dense layer configs.

    Strategy: open the .keras zip archive, strip 'quantization_config' from
    config.json recursively, write a patched temp archive, load from that.
    Fully stateless - no monkey-patching, safe to call multiple times.
    """
    model_path = str(model_path)

    def _strip(obj):
        if isinstance(obj, dict):
            return {k: _strip(v) for k, v in obj.items() if k != 'quantization_config'}
        if isinstance(obj, list):
            return [_strip(i) for i in obj]
        return obj

    with zipfile.ZipFile(model_path, 'r') as src:
        names = src.namelist()
        cfg = json.loads(src.read('config.json').decode('utf-8'))
        patched_cfg = json.dumps(_strip(cfg))

        tmp_fd, tmp_path = tempfile.mkstemp(suffix='.keras')
        os.close(tmp_fd)
        try:
            with zipfile.ZipFile(tmp_path, 'w', zipfile.ZIP_DEFLATED) as dst:
                for item in names:
                    dst.writestr(item, patched_cfg if item == 'config.json' else src.read(item))
            return keras.models.load_model(tmp_path, compile=False)
        finally:
            os.unlink(tmp_path)


def load_test_generator():
    test_csv   = './dataset/Test.csv'
    images_dir = './dataset/images/'
    print(f"Loading test data from: {test_csv}")
    if not os.path.exists(test_csv):
        raise FileNotFoundError(f"Test.csv not found at {test_csv}")
    df = pd.read_csv(test_csv)
    df['image_name'] = df['image_name'].astype(str)
    return DataGenerator(
        data_frame=df,
        y=IMG_DIM[0], x=IMG_DIM[1], target_channels=IMG_DIM[2],
        y_cols=CLASS_NAMES, batch_size=32,
        path_to_img=images_dir, shuffle=False,
        data_augmentation=False, mode='custom',
    )


def evaluate_checkpoint(model_path, model_name, stage_name):
    print(f"\n{'='*70}")
    print(f"Evaluating: {model_name} - {stage_name}")
    print(f"Model: {model_path}")
    print(f"{'='*70}")

    if not Path(model_path).exists():
        print(f"❌ Model not found: {model_path}")
        return None

    try:
        print("Loading model...")
        model = load_model_compat(model_path)
        model.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])
        print("✅ Model loaded")

        # Collect ground-truth labels from CSV directly (avoid generator cycling bug)
        # Use CLASS_NAMES=['NC','G3','G5','G4'] to match training class order
        print("Collecting labels...")
        test_df = pd.read_csv('./dataset/Test.csv')
        y_true = np.argmax(test_df[CLASS_NAMES].values.astype(np.float32), axis=1)
        n_samples = len(y_true)
        print(f"   {n_samples} samples, class distribution: { {CLASS_NAMES[i]: int((y_true==i).sum()) for i in range(4)} }")

        # Run inference in one pass (avoids tf.function retracing)
        # model.predict over ceil(n/32)*32 rows — trim back to n_samples
        print("Generating predictions...")
        test_gen_pred = load_test_generator()
        y_proba = model.predict(test_gen_pred, verbose=1)[:n_samples]
        y_pred = np.argmax(y_proba, axis=1)

        print(f"   y_true: {y_true.shape}, y_pred: {y_pred.shape}, y_proba: {y_proba.shape}")

        print("Calculating metrics...")
        report = classification_report(
            y_true, y_pred, target_names=CLASS_NAMES,
            output_dict=True, zero_division=0,
        )
        kappa = cohen_kappa_score(y_true, y_pred)

        auc_scores = {}
        try:
            for i, cls in enumerate(CLASS_NAMES):
                auc_scores[cls] = roc_auc_score((y_true == i).astype(int), y_proba[:, i])
        except Exception as e:
            print(f"Warning: AUC skipped: {e}")
            auc_scores = {c: 0.0 for c in CLASS_NAMES}

        cm = confusion_matrix(y_true, y_pred)
        result = {
            'model': model_name, 'stage': stage_name,
            'accuracy': report['accuracy'],
            'macro_f1': report['macro avg']['f1-score'],
            'weighted_f1': report['weighted avg']['f1-score'],
            'kappa': kappa, 'n_samples': n_samples,
            'confusion_matrix': cm.tolist(), 'per_class': {},
        }
        for cls in CLASS_NAMES:
            if cls in report:
                result['per_class'][cls] = {
                    'precision': report[cls]['precision'],
                    'recall':    report[cls]['recall'],
                    'f1':        report[cls]['f1-score'],
                    'support':   report[cls]['support'],
                    'auc':       auc_scores.get(cls, 0.0),
                }

        print(f"\n✅ Evaluation Complete:")
        print(f"   Accuracy: {result['accuracy']:.1%}")
        print(f"   Macro F1: {result['macro_f1']:.3f}")
        print(f"   Kappa:    {result['kappa']:.3f}")
        print(f"   Samples:  {result['n_samples']}")
        print(f"\n   Per-class Recall:")
        for cls in CLASS_NAMES:
            if cls in result['per_class']:
                print(f"      {cls}: {result['per_class'][cls]['recall']:.1%}")
        return result

    except Exception as e:
        print(f"❌ Error evaluating {model_name} - {stage_name}: {e}")
        import traceback; traceback.print_exc()
        return None


# Sanity-check: verify class order and label distribution
test_df_check = pd.read_csv('./dataset/Test.csv')
y_check = np.argmax(test_df_check[CLASS_NAMES].values.astype(np.float32), axis=1)
print(f"\n✅ Class order (must match training): {CLASS_NAMES}")
print(f"✅ Label sanity-check: {len(y_check)} samples")
print(f"   Distribution: { {CLASS_NAMES[i]: int((y_check==i).sum()) for i in range(4)} }")
test_gen = load_test_generator()
print(f"✅ Test generator loaded  ({len(test_gen)} batches × 32 = {len(test_gen)*32} slots for {len(y_check)} samples)")
print(f"✅ Evaluation function ready")


In [ ]:
# Evaluate all stage checkpoints
results = []

print("\n" + "="*70)
print("EVALUATING ALL STAGE 1 AND STAGE 2 CHECKPOINTS")
print("="*70)

# Baseline (no stages - single model)
baseline_dir = Path(RUNS_DIR) / 'baseline_20260329_110621' / 'baseline'
if baseline_dir.exists():
    result = evaluate_checkpoint(
        baseline_dir / 'best_baseline_classifier.keras',
        'Baseline (No SSL)',
        'Single Stage'
    )
    if result:
        results.append(result)

# CAE
cae_dir = Path(RUNS_DIR) / 'cae_20260329_114825'
if cae_dir.exists():
    # Stage 1 (frozen)
    result = evaluate_checkpoint(
        cae_dir / 'best_model_stage1.keras',
        'CAE-SSL',
        'Stage 1 (Frozen)'
    )
    if result:
        results.append(result)
    
    # Stage 2 (end-to-end)
    result = evaluate_checkpoint(
        cae_dir / 'best_model_fine_tuned.keras',
        'CAE-SSL',
        'Stage 2 (End-to-End)'
    )
    if result:
        results.append(result)

# SimCLR
simclr_dir = Path(RUNS_DIR) / 'simclr_20260329_122931' / 'simclr'
if simclr_dir.exists():
    # Stage 1 (frozen)
    result = evaluate_checkpoint(
        simclr_dir / 'best_simclr_classifier.keras',
        'SimCLR-SSL',
        'Stage 1 (Frozen)'
    )
    if result:
        results.append(result)
    
    # Stage 2 (end-to-end)
    result = evaluate_checkpoint(
        simclr_dir / 'best_simclr_fine_tuned.keras',
        'SimCLR-SSL',
        'Stage 2 (End-to-End)'
    )
    if result:
        results.append(result)

# MoCo (check for separate stage checkpoints)
moco_dir = Path(RUNS_DIR) / 'moco_20260329_024342' / 'output' / 'moco'
if moco_dir.exists():
    # Check selection summary
    selection_summary = moco_dir / 'selection_summary.json'
    if selection_summary.exists():
        print("\n" + "="*70)
        print("MoCo Stage Information (from selection_summary.json)")
        print("="*70)
        with open(selection_summary) as f:
            moco_info = json.load(f)
        print(json.dumps(moco_info, indent=2))
        print("\n⚠️  Note: MoCo stage-specific checkpoints not separately saved.")
        print("    Using final selected best model (Stage 2).\n")
    
    # Evaluate best overall model
    result = evaluate_checkpoint(
        moco_dir / 'best_moco_fine_tuned.keras',
        'MoCo-SSL',
        'Stage 2 (End-to-End) [Selected]'
    )
    if result:
        results.append(result)

print(f"\n✅ Evaluation complete. Total results: {len(results)}")

In [ ]:
# Generate comparison table
print("\n" + "="*70)
print("STAGE 1 vs STAGE 2 COMPARISON TABLE")
print("="*70 + "\n")

# Organize by model
by_model = {}
for result in results:
    model_name = result['model']
    if model_name not in by_model:
        by_model[model_name] = {}
    by_model[model_name][result['stage']] = result

# Create comparison rows
comparison_rows = []

for model_name in ['CAE-SSL', 'SimCLR-SSL', 'MoCo-SSL', 'Baseline (No SSL)']:
    if model_name not in by_model:
        continue
    
    stages = by_model[model_name]
    
    if 'Stage 1 (Frozen)' in stages and 'Stage 2 (End-to-End)' in stages:
        stage1 = stages['Stage 1 (Frozen)']
        stage2 = stages['Stage 2 (End-to-End)']
        
        # Calculate improvements
        acc_improvement = (stage2['accuracy'] - stage1['accuracy']) * 100
        kappa_improvement = stage2['kappa'] - stage1['kappa']
        f1_improvement = stage2['macro_f1'] - stage1['macro_f1']
        
        comparison_rows.append({
            'Model': model_name,
            'Stage 1 Acc': f"{stage1['accuracy']:.1%}",
            'Stage 2 Acc': f"{stage2['accuracy']:.1%}",
            'Acc Δ': f"{acc_improvement:+.1f}%",
            'Stage 1 F1': f"{stage1['macro_f1']:.3f}",
            'Stage 2 F1': f"{stage2['macro_f1']:.3f}",
            'F1 Δ': f"{f1_improvement:+.3f}",
            'Stage 1 κ': f"{stage1['kappa']:.3f}",
            'Stage 2 κ': f"{stage2['kappa']:.3f}",
            'κ Δ': f"{kappa_improvement:+.3f}",
            'Better Stage': '🟢 Stage 2' if stage2['kappa'] > stage1['kappa'] else '🔴 Stage 1',
            'n_samples': stage2['n_samples']
        })
    
    elif 'Single Stage' in stages:
        stage = stages['Single Stage']
        comparison_rows.append({
            'Model': model_name,
            'Stage 1 Acc': 'N/A',
            'Stage 2 Acc': f"{stage['accuracy']:.1%}",
            'Acc Δ': 'N/A',
            'Stage 1 F1': 'N/A',
            'Stage 2 F1': f"{stage['macro_f1']:.3f}",
            'F1 Δ': 'N/A',
            'Stage 1 κ': 'N/A',
            'Stage 2 κ': f"{stage['kappa']:.3f}",
            'κ Δ': 'N/A',
            'Better Stage': 'N/A (No SSL)',
            'n_samples': stage['n_samples']
        })
    
    elif 'Stage 2 (End-to-End) [Selected]' in stages:
        # MoCo special case
        stage2 = stages['Stage 2 (End-to-End) [Selected]']
        comparison_rows.append({
            'Model': model_name,
            'Stage 1 Acc': 'See val_loss*',
            'Stage 2 Acc': f"{stage2['accuracy']:.1%}",
            'Acc Δ': 'N/A',
            'Stage 1 F1': 'See val_loss*',
            'Stage 2 F1': f"{stage2['macro_f1']:.3f}",
            'F1 Δ': 'N/A',
            'Stage 1 κ': 'See val_loss*',
            'Stage 2 κ': f"{stage2['kappa']:.3f}",
            'κ Δ': 'N/A',
            'Better Stage': '🟢 Stage 2 (selected)',
            'n_samples': stage2['n_samples']
        })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))
print("\n* MoCo: Stage 1 val_loss=0.265, Stage 2 val_loss=0.121 (54.3% improvement)")

# Save to Drive
output_dir = Path(DRIVE_ROOT) / 'runs' / 'phase2_comparison'
output_dir.mkdir(exist_ok=True, parents=True)

comparison_csv = output_dir / 'stage_comparison_table.csv'
comparison_df.to_csv(comparison_csv, index=False)
print(f"\n✅ Saved to: {comparison_csv}")

In [ ]:
# Generate per-class comparison
print("\n" + "="*70)
print("PER-CLASS STAGE COMPARISON")
print("="*70 + "\n")

for model_name in ['CAE-SSL', 'SimCLR-SSL']:
    if model_name not in by_model:
        continue
    
    stages = by_model[model_name]
    
    if 'Stage 1 (Frozen)' in stages and 'Stage 2 (End-to-End)' in stages:
        stage1 = stages['Stage 1 (Frozen)']
        stage2 = stages['Stage 2 (End-to-End)']
        
        print(f"\n{model_name}:")
        print("="*70)
        
        per_class_rows = []
        for class_name in CLASS_NAMES:
            if class_name in stage1['per_class'] and class_name in stage2['per_class']:
                s1 = stage1['per_class'][class_name]
                s2 = stage2['per_class'][class_name]
                
                per_class_rows.append({
                    'Class': class_name,
                    'Stage 1 Recall': f"{s1['recall']:.1%}",
                    'Stage 2 Recall': f"{s2['recall']:.1%}",
                    'Recall Δ': f"{(s2['recall']-s1['recall'])*100:+.1f}%",
                    'Stage 1 F1': f"{s1['f1']:.3f}",
                    'Stage 2 F1': f"{s2['f1']:.3f}",
                    'F1 Δ': f"{s2['f1']-s1['f1']:+.3f}",
                    'Stage 1 AUC': f"{s1['auc']:.3f}",
                    'Stage 2 AUC': f"{s2['auc']:.3f}",
                    'AUC Δ': f"{s2['auc']-s1['auc']:+.3f}",
                })
        
        per_class_df = pd.DataFrame(per_class_rows)
        print(per_class_df.to_string(index=False))
        print()

print("\n✅ Per-class comparison complete")

In [ ]:
# Save detailed results as JSON
results_json = output_dir / 'stage_comparison_detailed_results.json'
with open(results_json, 'w') as f:
    json.dump(results, f, indent=2)

print(f"✅ Detailed results saved to: {results_json}")
print(f"\nAll outputs saved to: {output_dir}")
print("\nGenerated files:")
for file in sorted(output_dir.glob('*')):
    print(f"   - {file.name}")

In [ ]:
# Key findings summary
print("\n" + "="*70)
print("KEY FINDINGS: FROZEN vs END-TO-END FINE-TUNING")
print("="*70 + "\n")

# Analyze improvements
for model_name in ['CAE-SSL', 'SimCLR-SSL']:
    if model_name not in by_model:
        continue
    
    stages = by_model[model_name]
    
    if 'Stage 1 (Frozen)' in stages and 'Stage 2 (End-to-End)' in stages:
        stage1 = stages['Stage 1 (Frozen)']
        stage2 = stages['Stage 2 (End-to-End)']
        
        acc_delta = (stage2['accuracy'] - stage1['accuracy']) * 100
        kappa_delta = stage2['kappa'] - stage1['kappa']
        
        print(f"\n{model_name}:")
        if acc_delta > 0:
            print(f"   ✅ End-to-end fine-tuning HELPED: {acc_delta:+.1f}% accuracy")
        else:
            print(f"   ⚠️  End-to-end fine-tuning HURT: {acc_delta:+.1f}% accuracy")
        
        print(f"   κ change: {kappa_delta:+.3f}")
        
        # Check G5 improvements
        if 'G5' in stage1['per_class'] and 'G5' in stage2['per_class']:
            g5_recall_delta = (stage2['per_class']['G5']['recall'] - 
                              stage1['per_class']['G5']['recall']) * 100
            if g5_recall_delta > 0:
                print(f"   📈 G5 recall improved by {g5_recall_delta:+.1f}%")
            elif g5_recall_delta < 0:
                print(f"   📉 G5 recall decreased by {g5_recall_delta:.1f}%")

print("\n" + "="*70)
print("✅ PHASE 2 STAGE COMPARISON COMPLETE")
print("="*70)